# 🩺 Troubleshooting

Something not working? You're in the right place. Work through this notebook **top to bottom**: each section checks one thing and tells you how to fix it.

**First, run the full diagnostics** from a terminal in the repo root. It checks everything at once and writes `setup/report.txt`:
```bash
uv run setup/diagnostics.py
```
If even that fails, try `python setup/diagnostics.py` (Windows) or `python3 setup/diagnostics.py` (Mac/Linux).

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">If this notebook won't run at all, the kernel is probably wrong. Start with section 1 below; it's the most common problem in every course.</span>
        </td>
    </tr>
</table>

## 1. Is this notebook using the course's `.venv` kernel?

**Symptoms:** `ModuleNotFoundError: No module named 'langchain'` (or `dotenv`, `gradewise`…), even though `uv sync` worked.

In [ ]:
import sys

print(sys.executable)
print("✅ Correct kernel" if ".venv" in sys.executable else "❌ Wrong kernel: see the fix below")

**Fix:** top-right of the notebook → **Select Kernel** → **Python Environments…** → choose the `.venv` inside this repo.
- `.venv` not listed? Make sure you opened the **repo folder** (File → Open Folder), not a single file. Then <kbd>Ctrl/Cmd</kbd>+<kbd>Shift</kbd>+<kbd>P</kbd> → **Developer: Reload Window**.
- Still missing? In a terminal in the repo root, run `uv sync`, then reload the window.
- An Anaconda `(base)` prompt? Run `conda deactivate` first.

## 2. Is my `.env` file being found, and is the key well-formed?

**Symptoms:** "OpenAI API Key not set", or `MissingAPIKeyError`, or a 401 error from OpenAI.

In [ ]:
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "setup" else Path.cwd()
env_file = repo_root / ".env"
print("Looking for:", env_file)

if env_file.exists():
    print("✅ .env exists")
    for line in env_file.read_text(encoding="utf-8").splitlines():
        if line.startswith("OPENAI_API_KEY"):
            value = line.split("=", 1)[1]
            if not value:
                print("❌ OPENAI_API_KEY is empty: paste your key and SAVE")
            elif value != value.strip() or value[0] in "'\"":
                print("⚠️  Remove quotes/spaces around the key")
            elif not value.startswith("sk-"):
                print("⚠️  The key should start with sk-")
            else:
                print(f"✅ Key looks well-formed: {value[:8]}...")
elif (repo_root / ".env.txt").exists():
    print("❌ Found .env.txt: Windows hid the extension. Rename it to exactly .env")
else:
    print("❌ No .env file. In the repo root run:  copy .env.example .env  (Windows)  or  cp .env.example .env")

**Remember:** after editing `.env`, **save it**, then restart the notebook kernel (or re-run the cell containing `load_dotenv(override=True)`).

## 3. Can I reach the model provider? (network, VPN, proxy, SSL)

**Symptoms:** calls hang, `APIConnectionError`, `SSL: CERTIFICATE_VERIFY_FAILED`.

In [ ]:
import socket

for host in ["api.openai.com", "api.anthropic.com", "generativelanguage.googleapis.com", "api.smith.langchain.com"]:
    try:
        socket.create_connection((host, 443), timeout=5).close()
        print(f"✅ {host}")
    except OSError as e:
        print(f"❌ {host}: {e}")

**Fixes:**
- Pause your **VPN** and try again; some block AI APIs.
- On a **corporate network**, you may need the proxy settings from IT: set `HTTPS_PROXY=http://proxy:port` in `.env`.
- `CERTIFICATE_VERIFY_FAILED` on a corporate network usually means TLS inspection: ask IT for the root certificate, then set `SSL_CERT_FILE=/path/to/cert.pem` in `.env`.
- Or switch to free local models that need no network: `setup/SWAP_PROVIDER.md`, Option C.

## 4. "An Application Control policy has blocked this file" (Windows)

**Symptoms:** `os error 4551`, `WinError 4551`, or a package that fails to import with a confusing `ModuleNotFoundError` even though it's installed.

**Why:** Windows **Smart App Control** (on many new Windows 11 machines) and corporate **Application Control** policies block programs and Python extension files (`.pyd`, `.exe`) that are too new to have built a "reputation". Brand-new package releases are the usual victims.

The cell below tries to import every native package the course uses and tells you which, if any, are blocked:

In [ ]:
import importlib

for module in ["pymupdf", "chromadb", "pydantic_core", "tiktoken", "numpy", "PIL"]:
    try:
        importlib.import_module(module)
        print(f"✅ {module}")
    except Exception as e:
        print(f"❌ {module}: {type(e).__name__}: {e}")

**Fixes:**
1. **`git pull` and `uv sync`:** when I learn a version is blocked, I pin the course to one that works (that's why `pymupdf` is pinned to 1.25.x).
2. If a **program** like `gradewise.exe` is blocked, run it as a module instead: `uv run python -m gradewise …`
3. If a **Python version** is blocked, recreate the environment on another 3.12 release: delete the `.venv` folder, then run `uv venv --python 3.12.12` followed by `uv sync`.
4. On a work laptop, ask IT to allow Python development, or use a personal machine.
5. *(Personal machines only; your choice.)* Smart App Control can be turned off in **Windows Security → App & browser control**. ⚠️ It can't be turned back on without reinstalling Windows, so I don't recommend this lightly.

## 5. `uv` is not recognised

**Symptoms:** `uv : The term 'uv' is not recognized…` or `command not found: uv`.

**Fix:** close **all** terminals (and ideally Cursor/VS Code itself), then reopen. The installer updates your PATH, but only new terminals see it. Still missing? Re-run the install command from Part 3 of your setup guide.

## 6. Path too long / `[Errno 2] No such file or directory` during `uv sync` (Windows)

**Why:** the 260-character path limit. **Fix:** enable long paths (Windows gotcha 3 in `SETUP-PC.md`), restart, and keep the repo in a short path like `C:\Users\you\projects\`.

## 7. `RateLimitError` / HTTP 429 / "insufficient_quota"

**Why:** almost always **no credit** on the OpenAI account (new accounts start at $0), or a very low rate-limit tier.

**Fix:** add credit at [platform.openai.com/settings/organization/billing](https://platform.openai.com/settings/organization/billing), wait a minute, try again. Running many calls in a loop? Add `BUDGET_MODE=true` to `.env`, or slow down.

## 8. `UnicodeEncodeError: 'charmap' codec can't encode character` (Windows)

**Why:** older Windows terminals can't print emoji. GradeWise fixes this automatically when you use `gradewise.console`. In your own scripts, add these lines at the top:
```python
import sys
sys.stdout.reconfigure(encoding="utf-8", errors="replace")
```

## 9. Model not found / "The model `…` does not exist"

**Why:** model names change over time, or your account doesn't have access to that model yet. **Fix:** check `CHAT_MODEL`, `SMART_MODEL`, `VISION_MODEL` in `.env` against your provider's model list (links in `setup/SWAP_PROVIDER.md`). `git pull` also brings updated defaults, announced in `CHANGELOG.md`.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ec4899;">Still stuck? Let's fix it together</h2>
            <span style="color:#ec4899;">Still stuck? Please post in the Udemy Q&A for the setup lecture, attach <code>setup/report.txt</code> (it contains no secrets), and tell me what you've tried. I really want you to get going, so don't suffer in silence!</span>
        </td>
    </tr>
</table>